# Sprint 2: Entrenamiento — 4 clases, finetune full

Este notebook instala los archivos del Sprint 2 y entrena el modelo.

**Cambios respecto a Sprint 1:**
- 4 clases: No Finding / Cardiomegaly / Effusion / Infiltration.
- `finetune_mode: "full"` — se entrenan los 6.9M parámetros.
- `learning_rate: 1e-5` — 100x más bajo que Sprint 1.
- LR scheduler: ReduceLROnPlateau (reduce LR si val_loss estanca).
- Class weights: compensan desbalance residual.
- 20 epochs, patience=5.
- Dataset: ~1,300 imágenes (vs 279 del Sprint 1).

**Runtime:** T4 GPU obligatorio.
**Tiempo estimado:** 15-30 min para 20 epochs.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys, time, os
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"torch: {torch.__version__}")


In [ ]:
!pip install -q torchxrayvision scipy grad-cam
print("deps OK")


## 2. Verificar splits Sprint 2

In [ ]:
import pandas as pd

processed_s2 = f'{PROJECT_ROOT}/data/processed_s2'
CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

print("SPLITS SPRINT 2:")
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed_s2}/{split}.csv')
    counts = {CLASS_NAMES[l]: (df['label']==l).sum() for l in range(4)}
    print(f"  {split:5s}: {len(df):4d} imgs | {counts}")


## 3. Instalar archivos del Sprint 2

In [ ]:
from pathlib import Path


In [ ]:
# sprint2.yaml
content = r'''# configs/sprint2.yaml
# Sprint 2 - Baseline CNN optimizado, 4 clases, finetune full
# ─────────────────────────────────────────────────────────────
# Clases:
#   0 - No Finding
#   1 - Cardiomegaly   (insuficiencia cardíaca)
#   2 - Infiltration   (TB pulmonar / neumonía) ← más relevante en Loayza
#   3 - Effusion       (derrame pleural)
#
# Justificación clínica (Hospital Nacional Arzobispo Loayza):
#   Las 3 clases positivas cubren las patologías más frecuentes en el
#   servicio de Neumología / Medicina Interna del HNAL, donde la TB,
#   la neumonía y la insuficiencia cardíaca representan la mayor carga
#   asistencial (MINSA 2024, Lima concentra 44% de casos TB nacionales).
#
# Diferencias respecto a Sprint 1 (baseline.yaml):
#   - num_classes: 2 → 4
#   - finetune_mode: "head" → "full"
#   - learning_rate: 0.001 → 0.00001  (10x más bajo para full fine-tuning)
#   - LR scheduler activado (ReduceLROnPlateau)
#   - class_weights activado (inversamente proporcional a frecuencia)
#   - epochs: 10 → 20
#   - patience: 3 → 5

project:
  name: "tesis-cxr-sprint2"
  sprint: 2
  seed: 42

paths:
  project_root: "/content/drive/MyDrive/Tesis_CXR"
  data_raw: "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed_s2"
  experiments: "/content/drive/MyDrive/Tesis_CXR/experiments_s2"
  checkpoints: "/content/drive/MyDrive/Tesis_CXR/experiments_s2/checkpoints"
  logs: "/content/drive/MyDrive/Tesis_CXR/experiments_s2/logs"

dataset:
  name: "nih"
  csv_filename: "Data_Entry_subset_local.csv"
  images_subdir: "images"
  views: ["PA", "AP"]

  # 4 clases con jerarquía de prioridad clínica
  # Si una imagen tiene múltiples labels, se asigna el de mayor prioridad.
  # Prioridad: Infiltration > Effusion > Cardiomegaly > No Finding
  classes:
    - name: "No Finding"
      label: 0
      priority: 0
    - name: "Cardiomegaly"
      label: 1
      priority: 1
    - name: "Effusion"
      label: 2
      priority: 2
    - name: "Infiltration"
      label: 3
      priority: 3   # prioridad más alta: TB/neumonía

  negative_class: "No Finding"

# Balance de clases (a nivel paciente, 1 img/paciente)
# ratio = n_negativos / n_positivos_por_clase
# Con ratio=2 y 3 clases positivas balanceadas a la más pequeña:
#   n_cardio = n_effusion = n_infiltration = min(820, ...)
#   n_no_finding = n_cardio * 2
class_ratio: 2.0

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio: 0.15
  test_ratio: 0.15
  split_by: "patient_id"

model:
  backbone: "densenet121-res224-nih"
  num_classes: 4
  pretrained: true
  # "full": entrena TODOS los parámetros (6.9M)
  # Requiere LR más bajo para no destruir el preentrenamiento
  finetune_mode: "full"

training:
  batch_size: 32
  num_workers: 2
  epochs: 20
  learning_rate: 0.00001     # 10x más bajo que Sprint 1
  weight_decay: 0.0001
  optimizer: "adam"
  patience: 5                # más paciencia porque LR más bajo

  # LR scheduler: reduce LR si val loss no mejora
  scheduler:
    enabled: true
    type: "ReduceLROnPlateau"
    mode: "min"              # monitorear val loss
    factor: 0.5              # LR = LR * 0.5 cuando se activa
    patience: 2              # epochs sin mejora antes de reducir
    min_lr: 0.000001

  # Class weights: inversamente proporcionales a frecuencia
  # Evita que el modelo ignore las clases raras
  use_class_weights: true

  use_amp: true

evaluation:
  metrics: ["auc_macro", "auc_per_class", "accuracy", "precision_macro",
            "sensitivity_macro", "specificity_macro"]
  threshold: 0.5

gradcam:
  target_layer: "features.denseblock4"
  num_samples_per_class: 2   # 2 ejemplos por clase = 8 imágenes total
'''
path = f'{CODE_DIR}/configs/sprint2.yaml'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# prepare_splits_s2.py
content = r'''"""
scripts/prepare_splits_s2.py

Genera los splits train/val/test para el Sprint 2: 4 clases.

Clases:
  0 - No Finding
  1 - Cardiomegaly   (insuficiencia cardíaca)
  2 - Effusion       (derrame pleural)
  3 - Infiltration   (TB pulmonar / neumonía)

Diferencias respecto a prepare_splits.py (Sprint 1):
  - 4 clases en vez de 2.
  - Asignación de labels por jerarquía de prioridad clínica.
    Una imagen con multiples patologias recibe el label de mayor prioridad:
    Infiltration (3) > Effusion (2) > Cardiomegaly (1) > No Finding (0).
  - Balance a nivel paciente: la clase mas pequeña define el techo.
    Todas las clases positivas se submuestrean a ese techo.
    No Finding se submuestrea a ratio*techo.
  - Misma estrategia de split v4: StratifiedKFold a nivel paciente
    + 1 imagen aleatoria por paciente.

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/prepare_splits_s2.py
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config  # noqa: E402


# ---------------------------------------------------------------------------
# Clases y jerarquía
# ---------------------------------------------------------------------------
# Jerarquía de prioridad (mayor número = mayor prioridad).
# Si una imagen tiene múltiples patologías, se asigna la de mayor prioridad.
LABEL_HIERARCHY = {
    "Infiltration": 3,  # prioridad máxima: TB / neumonía
    "Effusion":     2,
    "Cardiomegaly": 1,
    "No Finding":   0,  # solo si no tiene ninguna de las anteriores
}

# Nombres de clase para logging
CLASS_NAMES = {0: "No Finding", 1: "Cardiomegaly",
               2: "Effusion",   3: "Infiltration"}


# ---------------------------------------------------------------------------
# Construcción de labels multi-clase con jerarquía
# ---------------------------------------------------------------------------
def build_multiclass_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Asigna un label entero 0-3 a cada imagen basándose en jerarquía clínica.

    Política:
      - Si 'Finding Labels' contiene 'Infiltration' → label=3
      - Si contiene 'Effusion'                      → label=2
      - Si contiene 'Cardiomegaly'                  → label=1
      - Si es exactamente 'No Finding'              → label=0
      - Cualquier otra combinación                  → NaN (se descarta)

    Nota: si una imagen tiene 'Effusion|Cardiomegaly', recibe label=2 (Effusion
    tiene prioridad). Esto refleja que en un contexto de triage el hallazgo
    de mayor urgencia define la clase.
    """
    df = df.copy()

    def assign_label(finding: str) -> float:
        for keyword, label in sorted(LABEL_HIERARCHY.items(),
                                     key=lambda x: -x[1]):
            if keyword == "No Finding":
                if finding == "No Finding":
                    return 0
            else:
                if keyword in finding:
                    return label
        return float('nan')  # ni "No Finding" ni ninguna de las 3 positivas

    df['label'] = df['Finding Labels'].apply(assign_label)
    return df


def filter_views(df: pd.DataFrame, views: list) -> pd.DataFrame:
    return df[df['View Position'].isin(views)].copy()


# ---------------------------------------------------------------------------
# Balance multi-clase a nivel paciente
# ---------------------------------------------------------------------------
def build_patient_level_label(df: pd.DataFrame) -> pd.Series:
    """
    Determina el label representativo de cada paciente usando la misma
    jerarquía: si un paciente tiene imágenes con distintos labels,
    se le asigna el de mayor prioridad.
    """
    return df.groupby('Patient ID')['label'].max().astype(int)


def balance_patients_multiclass(df: pd.DataFrame, ratio_negative: float,
                                seed: int, logger) -> pd.DataFrame:
    """
    Balancea el dataset a nivel paciente:
      1. Calcula el label representativo por paciente.
      2. Submuestrea cada clase positiva a la más pequeña (techo).
      3. Submuestrea No Finding a ratio_negative * techo.

    Args:
        df: DataFrame con columnas 'Patient ID', 'label'.
        ratio_negative: n_no_finding_pacientes = ratio_negative * n_min_positivo.
        seed: semilla para reproducibilidad.
    """
    rng = np.random.default_rng(seed)

    patient_labels = build_patient_level_label(df)

    # Separar pacientes por clase
    patients_by_class = {}
    for label in [0, 1, 2, 3]:
        pats = patient_labels[patient_labels == label].index.tolist()
        patients_by_class[label] = pats
        logger.info("  Clase %d (%s): %d pacientes",
                    label, CLASS_NAMES[label], len(pats))

    # Techo = mínimo entre las 3 clases positivas
    positive_counts = [len(patients_by_class[l]) for l in [1, 2, 3]]
    ceiling = min(positive_counts)
    logger.info("Techo de balance (clase más pequeña): %d pacientes", ceiling)

    # Submuestrear cada clase positiva al techo
    selected_patients = []
    for label in [1, 2, 3]:
        pats = patients_by_class[label]
        rng.shuffle(pats := list(pats))  # in-place shuffle
        selected = pats[:ceiling]
        selected_patients.extend(selected)
        logger.info("  Clase %d (%s): %d/%d pacientes seleccionados",
                    label, CLASS_NAMES[label], len(selected), len(pats))

    # Submuestrear No Finding
    n_neg_target = int(ceiling * ratio_negative)
    pats_neg = patients_by_class[0]
    rng.shuffle(pats_neg := list(pats_neg))
    selected_neg = pats_neg[:min(n_neg_target, len(pats_neg))]
    selected_patients.extend(selected_neg)
    logger.info("  Clase 0 (No Finding): %d pacientes seleccionados",
                len(selected_neg))

    # Filtrar df a los pacientes seleccionados
    selected_set = set(selected_patients)
    df_balanced = df[df['Patient ID'].isin(selected_set)].copy()
    df_balanced = df_balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    logger.info("Total tras balanceo: %d imágenes, %d pacientes",
                len(df_balanced), df_balanced['Patient ID'].nunique())
    return df_balanced


# ---------------------------------------------------------------------------
# Split estratificado a nivel paciente (misma lógica que Sprint 1 v4)
# ---------------------------------------------------------------------------
def stratified_patient_split(df: pd.DataFrame, train_ratio: float,
                             val_ratio: float, test_ratio: float,
                             seed: int, logger
                             ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    patient_labels = build_patient_level_label(df)
    logger.info("Pacientes únicos para split: %d", len(patient_labels))

    X_pat = patient_labels.index.to_numpy()
    y_pat = patient_labels.values

    # Paso 1: separar test
    n_splits_test = max(2, round(1.0 / test_ratio))
    skf_test = StratifiedKFold(n_splits=n_splits_test, shuffle=True,
                               random_state=seed)
    train_val_idx, test_idx = next(skf_test.split(X_pat, y_pat))

    test_patients = set(X_pat[test_idx])
    tv_patient_df = pd.DataFrame({
        'Patient ID': X_pat[train_val_idx],
        'patient_label': y_pat[train_val_idx],
    })

    # Paso 2: separar val de train+val
    val_ratio_rel = val_ratio / (train_ratio + val_ratio)
    n_splits_val = max(2, round(1.0 / val_ratio_rel))
    skf_val = StratifiedKFold(n_splits=n_splits_val, shuffle=True,
                              random_state=seed)
    X2 = tv_patient_df['Patient ID'].to_numpy()
    y2 = tv_patient_df['patient_label'].to_numpy()
    train_idx, val_idx = next(skf_val.split(X2, y2))

    train_patients = set(X2[train_idx])
    val_patients = set(X2[val_idx])

    logger.info("Split pacientes: train=%d, val=%d, test=%d",
                len(train_patients), len(val_patients), len(test_patients))

    df_train = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    df_val   = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    df_test  = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

    return df_train, df_val, df_test


def sample_one_image_per_patient(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Selecciona 1 imagen aleatoria por paciente (estrategia v4)."""
    rng = np.random.default_rng(seed)
    sampled = []
    for _, group in df.groupby('Patient ID'):
        idx = rng.integers(0, len(group))
        sampled.append(group.iloc[idx])
    return pd.DataFrame(sampled).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Reporte y validaciones
# ---------------------------------------------------------------------------
def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_patients = df['Patient ID'].nunique()
    logger.info("%-5s: %4d imgs | %3d pacientes", name.upper(), n, n_patients)
    for label, cname in CLASS_NAMES.items():
        count = (df['label'] == label).sum()
        pct = count / n * 100 if n else 0
        logger.info("  clase %d (%s): %d (%.1f%%)", label, cname, count, pct)


def check_no_leakage(train, val, test, logger) -> None:
    p_tr = set(train['Patient ID'])
    p_va = set(val['Patient ID'])
    p_te = set(test['Patient ID'])
    for name, overlap in [('train-val', p_tr & p_va),
                           ('train-test', p_tr & p_te),
                           ('val-test', p_va & p_te)]:
        if overlap:
            logger.error("LEAKAGE en %s: %d pacientes", name, len(overlap))
            raise AssertionError(f"Data leakage en {name}")
    logger.info("Sin leakage: pacientes disjuntos entre splits.")


def check_stratification(train, val, test, logger,
                         max_deviation: float = 10.0) -> None:
    combined = pd.concat([train, val, test], ignore_index=True)
    logger.info("Distribución global de clases:")
    for label, cname in CLASS_NAMES.items():
        global_pct = (combined['label'] == label).mean() * 100
        logger.info("  clase %d (%s): %.1f%% global", label, cname, global_pct)
        for sname, sdf in [("train", train), ("val", val), ("test", test)]:
            pct = (sdf['label'] == label).mean() * 100
            dev = abs(pct - global_pct)
            if dev > max_deviation:
                logger.warning("  %s tiene %.1f%% de clase %d "
                               "(desviación %.1fpp)", sname, pct, label, dev)
    logger.info("Verificación de estratificación completada.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint2.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])
    log_file = Path(cfg.paths['logs']) / "prepare_splits_s2.log"
    logger = get_logger("prepare_splits_s2", log_file=log_file)

    logger.info("=" * 60)
    logger.info("PREPARE SPLITS Sprint 2 — 4 clases")
    logger.info("Jerarquía: Infiltration(3) > Effusion(2) > "
                "Cardiomegaly(1) > No Finding(0)")
    logger.info("=" * 60)

    nih_dir = Path(cfg.paths['data_raw'])
    processed_dir = Path(cfg.paths['data_processed'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    # 1. Cargar CSV combinado de los 3 tarballs
    subset_csv = nih_dir / "Data_Entry_subset_local.csv"
    if not subset_csv.exists():
        raise FileNotFoundError(
            f"No se encontró {subset_csv}. "
            "Corre primero S2_01_download_more_data.ipynb."
        )

    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas
    df = filter_views(df, cfg.dataset['views'])
    logger.info("Tras filtrar vistas %s: %d filas",
                cfg.dataset['views'], len(df))

    # 3. Construir labels con jerarquía
    df = build_multiclass_labels(df)
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    logger.info("Tras construir labels: %d filas (descartadas %d sin clase)",
                len(df), n_before - len(df))

    for label, cname in CLASS_NAMES.items():
        n = (df['label'] == label).sum()
        logger.info("  %s: %d imágenes", cname, n)

    # 4. Balancear a nivel paciente
    ratio = cfg.get('class_ratio', 2.0)
    logger.info("Balanceando con ratio NF=%.1f * min_positivo...", ratio)
    df_balanced = balance_patients_multiclass(
        df, ratio_negative=ratio, seed=cfg.project['seed'], logger=logger
    )

    # 5. Split estratificado por paciente
    train_df, val_df, test_df = stratified_patient_split(
        df_balanced,
        train_ratio=cfg.splits['train_ratio'],
        val_ratio=cfg.splits['val_ratio'],
        test_ratio=cfg.splits['test_ratio'],
        seed=cfg.project['seed'],
        logger=logger,
    )

    # 6. Una imagen por paciente
    train_df = sample_one_image_per_patient(train_df, seed=cfg.project['seed'])
    val_df   = sample_one_image_per_patient(val_df,   seed=cfg.project['seed'] + 1)
    test_df  = sample_one_image_per_patient(test_df,  seed=cfg.project['seed'] + 2)

    # 7. Reportar y validar
    logger.info("-" * 60)
    report_split("train", train_df, logger)
    report_split("val",   val_df,   logger)
    report_split("test",  test_df,  logger)
    logger.info("-" * 60)
    check_no_leakage(train_df, val_df, test_df, logger)
    check_stratification(train_df, val_df, test_df, logger)

    # 8. Guardar
    cols = ['Image Index', 'Finding Labels', 'Patient ID', 'View Position', 'label']
    train_df[cols].to_csv(processed_dir / "train.csv", index=False)
    val_df[cols].to_csv(processed_dir   / "val.csv",   index=False)
    test_df[cols].to_csv(processed_dir  / "test.csv",  index=False)

    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(train_df))
    logger.info("  val.csv:   %d filas", len(val_df))
    logger.info("  test.csv:  %d filas", len(test_df))
    logger.info("=" * 60)
    logger.info("PREPARE SPLITS S2 OK")


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/prepare_splits_s2.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# metrics_s2.py
content = r'''"""
src/evaluation/metrics_s2.py

Métricas de evaluación para clasificación multi-clase (Sprint 2).

Métricas principales:
  - AUC macro: promedio de AUC one-vs-rest por clase.
  - AUC por clase: AUC individual para cada clase.
  - Accuracy (top-1).
  - Precision, Recall (Sensitivity), Specificity — macro averaged.

La estrategia one-vs-rest (OvR) para AUC es el estándar en papers
de clasificación multi-clase de CXR (Wang 2017, Rajpurkar 2017).
"""
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize


CLASS_NAMES = {
    0: "No Finding",
    1: "Cardiomegaly",
    2: "Effusion",
    3: "Infiltration",
}


# ---------------------------------------------------------------------------
# Conversión de logits a probabilidades
# ---------------------------------------------------------------------------
def logits_to_probs_multiclass(logits: torch.Tensor) -> torch.Tensor:
    """
    Convierte logits (B, num_classes) a probabilidades via softmax.
    Devuelve tensor (B, num_classes).
    """
    return torch.softmax(logits, dim=1)


# ---------------------------------------------------------------------------
# Métricas multi-clase
# ---------------------------------------------------------------------------
def compute_metrics_multiclass(y_true: np.ndarray | torch.Tensor,
                               y_prob: np.ndarray | torch.Tensor
                               ) -> dict:
    """
    Calcula métricas para clasificación multi-clase.

    Args:
        y_true: labels verdaderos (N,), enteros 0..C-1.
        y_prob: probabilidades softmax (N, C).

    Returns:
        dict con:
          auc_macro, auc_per_class,
          accuracy,
          precision_macro, sensitivity_macro, specificity_macro,
          confusion_matrix (C x C como lista).
    """
    y_true = _to_numpy(y_true).astype(int)
    y_prob = _to_numpy(y_prob).astype(float)

    n_classes = y_prob.shape[1]

    # AUC one-vs-rest por clase
    # Requiere que haya muestras de todas las clases presentes
    auc_per_class = []
    for c in range(n_classes):
        y_bin = (y_true == c).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            # Solo una clase presente → AUC no definida
            auc_per_class.append(float('nan'))
        else:
            auc_per_class.append(float(roc_auc_score(y_bin, y_prob[:, c])))

    # AUC macro: promedio excluyendo NaN
    valid_aucs = [a for a in auc_per_class if not np.isnan(a)]
    auc_macro = float(np.mean(valid_aucs)) if valid_aucs else float('nan')

    # Predicción top-1
    y_pred = np.argmax(y_prob, axis=1)

    # Accuracy
    accuracy = float(accuracy_score(y_true, y_pred))

    # Métricas por clase (OvR) → promediar (macro)
    precisions, sensitivities, specificities = [], [], []
    for c in range(n_classes):
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        tn = int(((y_pred != c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        precisions.append(prec)
        sensitivities.append(sens)
        specificities.append(spec)

    # Matriz de confusión
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    return {
        'auc_macro': auc_macro,
        'auc_per_class': auc_per_class,
        'accuracy': accuracy,
        'precision_macro': float(np.mean(precisions)),
        'sensitivity_macro': float(np.mean(sensitivities)),
        'specificity_macro': float(np.mean(specificities)),
        'precision_per_class': precisions,
        'sensitivity_per_class': sensitivities,
        'specificity_per_class': specificities,
        'confusion_matrix': cm.tolist(),
        'n_samples': int(len(y_true)),
    }


def _to_numpy(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


# ---------------------------------------------------------------------------
# Formateo para logging
# ---------------------------------------------------------------------------
def format_metrics(metrics: dict, class_names: dict = CLASS_NAMES) -> str:
    lines = [
        f"AUC_macro={metrics['auc_macro']:.4f} | "
        f"Acc={metrics['accuracy']:.4f} | "
        f"Prec_macro={metrics['precision_macro']:.4f} | "
        f"Sens_macro={metrics['sensitivity_macro']:.4f}",
    ]
    for i, auc in enumerate(metrics['auc_per_class']):
        name = class_names.get(i, f"clase_{i}")
        lines.append(f"  AUC {name}: {auc:.4f}")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    np.random.seed(42)
    n, c = 200, 4
    y_true = np.random.randint(0, c, size=n)
    y_prob = np.random.dirichlet(np.ones(c), size=n)
    # Mejorar un poco para que no sea puro ruido
    for i in range(n):
        y_prob[i, y_true[i]] += 0.5
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)

    m = compute_metrics_multiclass(y_true, y_prob)
    print("Smoke test metrics_s2.py:")
    print(format_metrics(m))
    print(f"\nMatriz de confusión:\n{np.array(m['confusion_matrix'])}")
'''
path = f'{CODE_DIR}/src/evaluation/metrics_s2.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# train_s2.py
content = r'''"""
src/training/train_s2.py

Loop de entrenamiento para Sprint 2: 4 clases, finetune full, LR scheduler,
class weights.

Diferencias respecto a Sprint 1 (train.py):
  - CrossEntropyLoss con class_weights para compensar desbalance residual.
  - ReduceLROnPlateau scheduler: reduce LR si val_loss no mejora.
  - Logging de AUC macro + AUC por clase (no solo AUC binario).
  - Compatibilidad con num_classes configurable (2, 3 o 4).
"""
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.evaluation.metrics_s2 import compute_metrics_multiclass, logits_to_probs_multiclass
from src.training.utils import save_checkpoint


# ---------------------------------------------------------------------------
# Una epoch de entrenamiento
# ---------------------------------------------------------------------------
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module,
                    device: torch.device,
                    use_amp: bool,
                    scaler: GradScaler | None) -> dict:
    model.train()
    losses = []
    all_probs = []
    all_labels = []

    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler is not None:
            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
        with torch.no_grad():
            probs = logits_to_probs_multiclass(logits.detach().float())
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    metrics = compute_metrics_multiclass(all_labels, all_probs)
    return {'loss': float(np.mean(losses)), 'metrics': metrics}


# ---------------------------------------------------------------------------
# Una epoch de validación
# ---------------------------------------------------------------------------
@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader,
                       criterion: nn.Module,
                       device: torch.device) -> dict:
    model.eval()
    losses = []
    all_probs = []
    all_labels = []

    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        losses.append(loss.item())
        probs = logits_to_probs_multiclass(logits.float())
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    metrics = compute_metrics_multiclass(all_labels, all_probs)
    return {'loss': float(np.mean(losses)), 'metrics': metrics}


# ---------------------------------------------------------------------------
# Calcular class weights desde el dataset de entrenamiento
# ---------------------------------------------------------------------------
def compute_class_weights(train_loader: DataLoader,
                          num_classes: int,
                          device: torch.device) -> torch.Tensor:
    """
    Calcula pesos inversamente proporcionales a la frecuencia de cada clase.
    Peso_clase = n_total / (n_clases * n_clase_i)
    """
    counts = torch.zeros(num_classes)
    for batch in train_loader:
        for label in batch['label']:
            counts[label.item()] += 1

    total = counts.sum()
    weights = total / (num_classes * counts)
    weights = weights / weights.sum() * num_classes  # normalizar
    return weights.to(device)


# ---------------------------------------------------------------------------
# Loop principal
# ---------------------------------------------------------------------------
def train_model_s2(model: nn.Module,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   cfg: Any,
                   device: torch.device,
                   logger) -> dict:
    """
    Entrena el modelo Sprint 2 con:
      - CrossEntropyLoss + class weights opcionales.
      - ReduceLROnPlateau scheduler.
      - Early stopping por val AUC macro.
      - Checkpointing del mejor modelo.
    """
    num_classes = cfg.model['num_classes']

    # Parámetros entrenables
    trainable = [p for p in model.parameters() if p.requires_grad]
    logger.info("Parámetros entrenables: %d tensors, %d scalars",
                len(trainable), sum(p.numel() for p in trainable))

    # Optimizer
    lr = cfg.training['learning_rate']
    wd = cfg.training.get('weight_decay', 0.0)
    optimizer = torch.optim.Adam(trainable, lr=lr, weight_decay=wd)
    logger.info("Optimizer: Adam | LR=%.2e | WD=%.2e", lr, wd)

    # Class weights
    use_weights = cfg.training.get('use_class_weights', False)
    if use_weights:
        logger.info("Calculando class weights...")
        weights = compute_class_weights(train_loader, num_classes, device)
        logger.info("Class weights: %s", weights.tolist())
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        criterion = nn.CrossEntropyLoss()
        logger.info("Sin class weights.")

    # LR Scheduler
    scheduler_cfg = cfg.training.get('scheduler', {})
    use_scheduler = scheduler_cfg.get('enabled', False)
    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode=scheduler_cfg.get('mode', 'min'),
            factor=scheduler_cfg.get('factor', 0.5),
            patience=scheduler_cfg.get('patience', 2),
            min_lr=scheduler_cfg.get('min_lr', 1e-6),
        )
        logger.info("Scheduler: ReduceLROnPlateau | factor=%.1f | patience=%d",
                    scheduler_cfg.get('factor', 0.5),
                    scheduler_cfg.get('patience', 2))

    # AMP
    use_amp = cfg.training.get('use_amp', False) and torch.cuda.is_available()
    scaler = GradScaler(device='cuda') if use_amp else None
    logger.info("Mixed precision (AMP): %s", use_amp)

    epochs = cfg.training['epochs']
    patience = cfg.training.get('patience', 5)

    ckpt_dir = Path(cfg.paths['checkpoints'])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = ckpt_dir / 'sprint2_best.pt'

    best_val_auc = -float('inf')
    best_epoch = -1
    best_state_dict = None
    epochs_no_improve = 0
    history = []

    logger.info("=" * 70)
    logger.info("Iniciando entrenamiento S2: %d epochs | patience=%d | "
                "num_classes=%d | finetune=%s",
                epochs, patience, num_classes, cfg.model['finetune_mode'])
    logger.info("=" * 70)

    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[0]['lr']

        train_res = train_one_epoch(model, train_loader, optimizer,
                                    criterion, device, use_amp, scaler)
        val_res = validate_one_epoch(model, val_loader, criterion, device)

        tr_m = train_res['metrics']
        va_m = val_res['metrics']

        # Step del scheduler sobre val_loss
        if scheduler is not None:
            scheduler.step(val_res['loss'])

        logger.info(
            "Epoch %2d/%d | LR=%.2e | "
            "train loss=%.4f AUC_macro=%.4f | "
            "val loss=%.4f AUC_macro=%.4f acc=%.4f",
            epoch, epochs, current_lr,
            train_res['loss'], tr_m['auc_macro'],
            val_res['loss'], va_m['auc_macro'], va_m['accuracy'],
        )

        # Log AUC por clase en val
        for i, auc in enumerate(va_m['auc_per_class']):
            logger.info("  val AUC clase %d: %.4f", i, auc)

        history.append({
            'epoch': epoch,
            'lr': current_lr,
            'train_loss': train_res['loss'],
            'train_metrics': tr_m,
            'val_loss': val_res['loss'],
            'val_metrics': va_m,
        })

        # Early stopping por AUC macro
        score = va_m['auc_macro']
        if score > best_val_auc:
            best_val_auc = score
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            logger.info("  → Nuevo mejor modelo (val AUC_macro=%.4f)", best_val_auc)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                logger.info("Early stopping: %d epochs sin mejora.", patience)
                break

    # Guardar mejor checkpoint
    if best_state_dict is not None:
        save_checkpoint({
            'epoch': best_epoch,
            'model_state_dict': best_state_dict,
            'best_val_auc_macro': best_val_auc,
            'num_classes': num_classes,
            'cfg_snapshot': {
                'backbone': cfg.model['backbone'],
                'num_classes': cfg.model['num_classes'],
                'finetune_mode': cfg.model['finetune_mode'],
            },
        }, best_ckpt_path)
        logger.info("Mejor modelo guardado: %s (epoch %d, AUC_macro=%.4f)",
                    best_ckpt_path, best_epoch, best_val_auc)

    logger.info("=" * 70)
    logger.info("Entrenamiento S2 terminado.")

    return {
        'history': history,
        'best_epoch': best_epoch,
        'best_val_auc_macro': best_val_auc,
        'checkpoint_path': str(best_ckpt_path),
    }
'''
path = f'{CODE_DIR}/src/training/train_s2.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# run_train_s2.py
content = r'''"""
scripts/run_train_s2.py

Entrypoint del entrenamiento Sprint 2 (4 clases, finetune full).

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/run_train_s2.py
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih import build_nih_datasets_from_processed
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.baseline import build_baseline_model
from src.training.train_s2 import train_model_s2
from src.training.utils import (
    get_device, get_logger, load_config, set_seed,
)


def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint2.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])

    log_file = Path(cfg.paths['logs']) / "train_s2.log"
    logger = get_logger("train_s2", log_file=log_file)

    logger.info("=" * 70)
    logger.info("RUN TRAIN Sprint 2 — 4 clases, finetune full")
    logger.info("Config: %s", config_path)
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s", device)
    if device.type == 'cuda':
        logger.info("GPU: %s", torch.cuda.get_device_name(0))

    # Datasets y loaders
    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm  = build_eval_transform(image_size=cfg.preprocessing['image_size'])

    # Los splits del Sprint 2 están en data/processed_s2/
    train_ds, val_ds, test_ds = build_nih_datasets_from_processed(
        processed_dir=cfg.paths['data_processed'],
        images_dir=str(Path(cfg.paths['data_raw']) / cfg.dataset['images_subdir']),
        train_transform=train_tfm,
        eval_transform=eval_tfm,
    )
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))

    train_loader = DataLoader(
        train_ds, batch_size=cfg.training['batch_size'], shuffle=True,
        num_workers=cfg.training.get('num_workers', 0),
        pin_memory=(device.type == 'cuda'),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.training['batch_size'], shuffle=False,
        num_workers=cfg.training.get('num_workers', 0),
        pin_memory=(device.type == 'cuda'),
    )

    # Modelo (4 clases, finetune full)
    model = build_baseline_model(cfg).to(device)
    logger.info("Modelo: %s | backbone=%s | mode=%s | num_classes=%d",
                model.__class__.__name__,
                cfg.model['backbone'],
                cfg.model['finetune_mode'],
                cfg.model['num_classes'])
    logger.info("  total params: %d | trainable: %d",
                model.total_params(), model.trainable_params())

    # Entrenamiento
    result = train_model_s2(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        cfg=cfg,
        device=device,
        logger=logger,
    )

    # Guardar history
    history_path = Path(cfg.paths['experiments']) / "history_s2.json"
    history_path.parent.mkdir(parents=True, exist_ok=True)
    with history_path.open('w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    logger.info("History guardado: %s", history_path)
    logger.info("RESUMEN: best epoch=%d | best val AUC_macro=%.4f",
                result['best_epoch'], result['best_val_auc_macro'])


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/run_train_s2.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# Agregar factory function a nih.py
nih_path = f'{CODE_DIR}/src/datasets/nih.py'
with open(nih_path) as f:
    nih_content = f.read()

addon = r'''
def build_nih_datasets_from_processed(
        processed_dir: str,
        images_dir: str,
        train_transform=None,
        eval_transform=None,
) -> tuple:
    """
    Crea los tres datasets leyendo CSVs desde un directorio de splits
    personalizado (permite Sprint 1 y Sprint 2 coexistir).
    """
    from pathlib import Path as _Path
    processed = _Path(processed_dir)
    imgs = _Path(images_dir)
    train_ds = NIHDataset(processed / "train.csv", imgs, train_transform)
    val_ds   = NIHDataset(processed / "val.csv",   imgs, eval_transform)
    test_ds  = NIHDataset(processed / "test.csv",  imgs, eval_transform)
    return train_ds, val_ds, test_ds
'''

if 'build_nih_datasets_from_processed' not in nih_content:
    with open(nih_path, 'a', encoding='utf-8') as f:
        f.write(addon)
    print("OK -> nih.py actualizado con build_nih_datasets_from_processed")
else:
    print("OK -> nih.py ya tiene build_nih_datasets_from_processed")


## 4. Crear carpeta experiments_s2

In [ ]:
import os
for folder in [
    f'{PROJECT_ROOT}/experiments_s2',
    f'{PROJECT_ROOT}/experiments_s2/checkpoints',
    f'{PROJECT_ROOT}/experiments_s2/logs',
    f'{PROJECT_ROOT}/experiments_s2/figures',
]:
    os.makedirs(folder, exist_ok=True)
    print(f'OK -> {folder}')


In [ ]:
# Celda A: Copiar imágenes a disco local de Colab
import shutil, os, time
from pathlib import Path

LOCAL_IMAGES = '/content/nih_images'
PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
CODE_DIR = f'{PROJECT_ROOT}/code'
DRIVE_IMAGES = f'{NIH_DIR}/images'

if not os.path.exists(LOCAL_IMAGES) or len(os.listdir(LOCAL_IMAGES)) < 1000:
    print(f'Copiando imágenes de Drive a disco local...')
    print(f'(tarda ~5-10 min pero el entrenamiento será mucho más rápido)')
    t0 = time.time()
    shutil.copytree(DRIVE_IMAGES, LOCAL_IMAGES, dirs_exist_ok=True)
    elapsed = time.time() - t0
    n = len(os.listdir(LOCAL_IMAGES))
    print(f'✓ {n:,} imágenes copiadas en {elapsed/60:.1f} min')
else:
    n = len(os.listdir(LOCAL_IMAGES))
    print(f'✓ Ya existen {n:,} imágenes en disco local')

print(f'LOCAL_IMAGES: {LOCAL_IMAGES}')

## 5. Ejecutar entrenamiento Sprint 2

La primera época tarda más porque carga los pesos preentrenados.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s2.py'],
    capture_output=True, text=True,
)
print('STDOUT:')
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print()
print('LOG:')
print(result.stderr[-6000:] if len(result.stderr) > 6000 else result.stderr)


## 6. Cargar history y plotear curvas

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'

if not __import__('os').path.exists(history_path):
    print("history_s2.json no encontrado. Revisa la celda anterior.")
else:
    with open(history_path) as f:
        data = json.load(f)

    history = data['history']
    epochs  = [h['epoch'] for h in history]
    t_loss  = [h['train_loss'] for h in history]
    v_loss  = [h['val_loss'] for h in history]
    t_auc   = [h['train_metrics']['auc_macro'] for h in history]
    v_auc   = [h['val_metrics']['auc_macro'] for h in history]
    t_acc   = [h['train_metrics']['accuracy'] for h in history]
    v_acc   = [h['val_metrics']['accuracy'] for h in history]

    print(f"Best epoch: {data['best_epoch']} | Best val AUC_macro: {data['best_val_auc_macro']:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (tr, va, title, ylabel) in zip(axes, [
        (t_loss, v_loss, 'Loss',     'Loss'),
        (t_auc,  v_auc,  'AUC Macro','AUC'),
        (t_acc,  v_acc,  'Accuracy', 'Accuracy'),
    ]):
        ax.plot(epochs, tr, 'o-', label='train', color='steelblue')
        ax.plot(epochs, va, 's-', label='val',   color='coral')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
        if ylabel != 'Loss':
            ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
            ax.set_ylim(0, 1.05)

    plt.suptitle(f"Sprint 2 — best val AUC_macro={data['best_val_auc_macro']:.4f} @ epoch {data['best_epoch']}")
    plt.tight_layout()
    plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/training_curves_s2.png', dpi=120)
    plt.show()


## 7. AUC por clase en el mejor epoch

In [ ]:
if 'history' in dir() and history:
    best_h = history[data['best_epoch'] - 1]
    print(f"Epoch {data['best_epoch']} — métricas de validación:")
    print(f"  AUC macro:    {best_h['val_metrics']['auc_macro']:.4f}")
    print(f"  Accuracy:     {best_h['val_metrics']['accuracy']:.4f}")
    print()
    for i, (auc, sens, spec) in enumerate(zip(
        best_h['val_metrics']['auc_per_class'],
        best_h['val_metrics']['sensitivity_per_class'],
        best_h['val_metrics']['specificity_per_class'],
    )):
        name = CLASS_NAMES.get(i, f'clase_{i}')
        print(f"  Clase {i} ({name}):")
        print(f"    AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f}")


## 8. Siguiente paso

Con el training terminado, vuelve al chat para recibir:
1. `src/evaluation/evaluate_s2.py` — evaluación sobre test con métricas multi-clase.
2. `scripts/run_eval_s2.py` — entrypoint.
3. Grad-CAM actualizado para 4 clases.
4. Comparación Sprint 1 vs Sprint 2.
